In [2]:
# 📦 Imports
import pandas as pd
import numpy as np
import os
import re
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms
from transformers import BertTokenizer, BertModel

C:\Users\shubh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ✅ Load and shuffle dataset
df = pd.read_csv("drive/MyDrive/TR-OP/tr_op_summary_5000.csv")
df = shuffle(df, random_state=42).reset_index(drop=True)

In [4]:
df.head(10)

,HTMLContent,InputURL,Info,ScreenshotPath,GoodorBad
0,"<!DOCTYPE html><html lang=""en""><head>\n <me...",https://berita-viral-malaysia.xx-idx.my.id/,https://berita-viral-malaysia.xx-idx.my.id/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0
1,"<!DOCTYPE html><html class=""""><head>\n <met...",https://tg.dev,https://core.telegram.org/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,1
2,"<html xml:lang=""it"" lang=""it""><head>\n<title>B...",https://certifica.91-92-243-223.cprapid.com/,https://certifica.91-92-243-223.cprapid.com/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0
3,"<!DOCTYPE html><html lang=""en""><head>\n\n <...",http://byronzepeda.com/99877e.html,http://byronzepeda.com/99877e.html,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0
4,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 S...",https://studfile.net,https://studfile.net/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,1
5,"<!DOCTYPE html><html dir=""ltr"" lang=""en""><head...",https://takealot.com,https://www.takealot.com/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,1
6,"<!DOCTYPE html><html lang=""zh-cn""><head><meta ...",https://blued.cn,https://www.blued.cn/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,1
7,"<!DOCTYPE html><html lang=""en""><head><meta htt...",https://ghfdddd.weebly.com/,https://ghfdddd.weebly.com/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0
8,"<!DOCTYPE html><html class=""no-js"" lang=""nl""><...",https://web11781.web09.bero-webspace.de/,https://web11781.web09.bero-webspace.de/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0
9,"<!DOCTYPE html><html lang=""en""><head><meta cha...",https://olrt-5dbf4.web.app/,https://olrt-5dbf4.web.app/,C:/Users/shubh/Desktop/PHISHING/drive/MyDrive/...,0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# ===== STEP 1: Split Data FIRST to avoid leakage =====
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42, shuffle=True)

In [7]:
# ===== STEP 2: URL Lexical features =====
def extract_url_features(url):
    return {
        'url_length': len(url),
        'num_digits': sum(c.isdigit() for c in url),
        'num_dots': url.count('.'),
        'num_slashes': url.count('/'),
        'num_hyphens': url.count('-'),
        'has_ip': int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', url))),
        'uses_https': int(url.startswith('https://'))
    }
url_train_feat = df_train['InputURL'].apply(lambda x: pd.Series(extract_url_features(x)))
url_test_feat = df_test['InputURL'].apply(lambda x: pd.Series(extract_url_features(x)))

In [10]:
url_test_feat

,url_length,num_digits,num_dots,num_slashes,num_hyphens,has_ip,uses_https
4022,56,1,2,3,0,0,1
5412,25,0,1,2,0,0,1
487,23,0,1,2,0,0,1
39,77,10,5,3,2,0,0
6795,93,16,2,8,1,0,0
...,...,...,...,...,...,...,...
1579,46,1,1,3,0,0,0
7905,37,2,2,4,0,0,1
6113,19,0,1,2,0,0,1
3532,20,1,1,2,0,0,1


In [11]:

# ===== STEP 3: TF-IDF vectors (fit ONLY on train) =====
html_vectorizer = TfidfVectorizer(max_features=8000)
url_vectorizer = TfidfVectorizer(max_features=1000)

html_vectorizer.fit(df_train["HTMLContent"].fillna(""))
url_vectorizer.fit(df_train["InputURL"].fillna(""))

X_html_train = html_vectorizer.transform(df_train["HTMLContent"]).toarray()
X_html_test = html_vectorizer.transform(df_test["HTMLContent"]).toarray()

X_url_train = url_vectorizer.transform(df_train["InputURL"]).toarray()
X_url_test = url_vectorizer.transform(df_test["InputURL"]).toarray()



In [12]:
X_url_train 

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(6993, 1000))

In [13]:
# ✅ BERT for HTML (pooler output)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

def get_bert_embedding(text):
    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.pooler_output.cpu().numpy().flatten()

X_html_train = np.vstack(df_train["HTMLContent"].fillna("").apply(get_bert_embedding).values)
X_html_test = np.vstack(df_test["HTMLContent"].fillna("").apply(get_bert_embedding).values)

In [14]:
X_html_test

array([[-0.6324692 , -0.4632316 , -0.86243564, ..., -0.83009577,
        -0.73590016,  0.5863291 ],
       [-0.7543807 , -0.55979586, -0.9590216 , ..., -0.9197812 ,
        -0.8272327 ,  0.77999246],
       [-0.6992817 , -0.5646301 , -0.90476334, ..., -0.92386335,
        -0.74450934,  0.7289237 ],
       ...,
       [-0.517168  , -0.40622407, -0.7873866 , ..., -0.84300286,
        -0.7071086 ,  0.5888102 ],
       [-0.6725738 , -0.44999376, -0.87425   , ..., -0.7980754 ,
        -0.7195468 ,  0.73217416],
       [-0.5638547 , -0.4704232 , -0.91397524, ..., -0.9034803 ,
        -0.65225685,  0.53318864]], shape=(2997, 768), dtype=float32)

In [15]:
# ===== STEP 4: Image embeddings (ResNet50) =====
class ScreenshotDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df["ScreenshotPath"].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            image = Image.open(path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image
        except:
            return torch.zeros(3, 224, 224)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# Train Images
img_train_ds = ScreenshotDataset(df_train, transform=transform)
train_loader = DataLoader(img_train_ds, batch_size=16, shuffle=False)

# Test Images
img_test_ds = ScreenshotDataset(df_test, transform=transform)
test_loader = DataLoader(img_test_ds, batch_size=16, shuffle=False)

# ResNet50 Embedding extraction
resnet = models.resnet50(pretrained=True).to(device)
resnet.fc = nn.Identity()
resnet.eval()

def extract_img_features(loader):
    features = []
    with torch.no_grad():
        for imgs in loader:
            imgs = imgs.to(device)
            feats = resnet(imgs).cpu().numpy()
            features.append(feats)
    return np.vstack(features)

X_img_train = extract_img_features(train_loader)
X_img_test = extract_img_features(test_loader)

C:\Users\shubh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\shubh\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [16]:
X_img_test[0]

array([0.30786136, 0.9120334 , 0.41771424, ..., 0.15260011, 0.29525343,
       0.08725972], shape=(2048,), dtype=float32)

In [11]:
# ✅ Combine All Features
X_train_combined = np.hstack([X_html_train, X_url_train, url_train_feat.values, X_img_train])
X_test_combined = np.hstack([X_html_test, X_url_test, url_test_feat.values, X_img_test])

# ✅ Standardize
scaler = StandardScaler()
X_train_combined = scaler.fit_transform(X_train_combined)
X_test_combined = scaler.transform(X_test_combined)

y_train = df_train["GoodorBad"].values
y_test = df_test["GoodorBad"].values

# ✅ Convert to tensors
train_dataset = TensorDataset(
    torch.tensor(X_train_combined, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [12]:
# ===== STEP 6: Neural Network classifier (PyTorch) =====
class PhishingNet(nn.Module):
    def __init__(self, input_size):
        super(PhishingNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

input_size = X_train_combined.shape[1]
model = PhishingNet(input_size).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_dataset = TensorDataset(
    torch.tensor(X_train_combined, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


In [13]:
# ===== STEP 7: Training =====
epochs = 15
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

print("✅ Model, vectorizer, and scaler saved.")

Epoch 1/15, Loss: 0.3264
Epoch 2/15, Loss: 0.1681
Epoch 3/15, Loss: 0.1058
Epoch 4/15, Loss: 0.0791
Epoch 5/15, Loss: 0.0575
Epoch 6/15, Loss: 0.0446
Epoch 7/15, Loss: 0.0415
Epoch 8/15, Loss: 0.0341
Epoch 9/15, Loss: 0.0321
Epoch 10/15, Loss: 0.0256
Epoch 11/15, Loss: 0.0277
Epoch 12/15, Loss: 0.0221
Epoch 13/15, Loss: 0.0246
Epoch 14/15, Loss: 0.0158
Epoch 15/15, Loss: 0.0151
✅ Model, vectorizer, and scaler saved.


In [14]:
import time  # Add this at the top of your script

# ===== STEP 8: Evaluation =====
model.eval()

start_time = time.time()  # Start timing

with torch.no_grad():
    y_pred_probs = model(torch.tensor(X_test_combined, dtype=torch.float32).to(device)).cpu().numpy()

end_time = time.time()  # End timing

inference_time = end_time - start_time
avg_time_per_sample = inference_time / len(X_test_combined)

y_pred = (y_pred_probs >= 0.5).astype(int)

print("\n✅ Final Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Phishing", "Benign"]))

print(f"\n🕒 Total Inference Time: {inference_time:.4f} seconds")
print(f"⏱️ Average Inference Time per Sample: {avg_time_per_sample:.6f} seconds")


✅ Final Accuracy: 0.9813146479813146
              precision    recall  f1-score   support

    Phishing       0.99      0.98      0.98      1508
      Benign       0.98      0.99      0.98      1489

    accuracy                           0.98      2997
   macro avg       0.98      0.98      0.98      2997
weighted avg       0.98      0.98      0.98      2997


🕒 Total Inference Time: 0.0135 seconds
⏱️ Average Inference Time per Sample: 0.000004 seconds
